<a href="https://colab.research.google.com/github/UmymaM/Speech-Emotion-Recognition/blob/main/src/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import zipfile as zf
with zf.ZipFile("/content/drive/MyDrive/TESS Toronto emotional speech set data-20250108T135115Z-001.zip","r") as zip_ref:
  zip_ref.extractall("/content")

In [32]:
import os
import numpy as np
import torch
import librosa
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, random_split

In [33]:
class EmotionDataset(Dataset):
  # defining a custom dataset class
    def __init__(self, data_path, emotions, transform=None):
        self.data_path = data_path #path of dataset
        self.emotions = emotions #list of emotions
        self.file_list = []
        self.labels = []
        self.transform = transform #object to perform transformations on data

        for idx, emotion in enumerate(emotions):
            emotion_folders = [f'YAF_{emotion}', f'OAF_{emotion}']
            for folder in emotion_folders:
                folder_path = os.path.join(data_path, folder)
                # joining data path with the name of the folder to get the complete folder path
                # e.g. "/content/TESS Toronto emotional speech set data/OAF_Fear/
                if os.path.exists(folder_path):
                    for file_name in os.listdir(folder_path):
                        file_path = os.path.join(folder_path, file_name)
                        # joining folder path with filenames to get the complete audio path
                        # "/content/TESS Toronto emotional speech set data/OAF_Fear/OAF_back_fear.wav"
                        self.file_list.append(file_path)
                        self.labels.append(idx)
                        # adding file paths and their respective labels to file_list and labels

    def __len__(self):
      #method to get the total number of audios
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        label = self.labels[idx]
        y, sr = librosa.load(file_path, sr=16000)
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
        max_length = 128
        pad_width = max_length - mel_spectrogram_db.shape[1]
        if pad_width > 0:
            mel_spectrogram_db = np.pad(mel_spectrogram_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mel_spectrogram_db = mel_spectrogram_db[:, :max_length]
        mel_spectrogram_3ch = np.repeat(mel_spectrogram_db[np.newaxis, :, :], 3, axis=0)
        return torch.tensor(mel_spectrogram_3ch, dtype=torch.float32), torch.tensor(label)

In [34]:
class EmotionRecognitionModel(nn.Module):
  #defining the emotion recognition model
    def __init__(self, num_classes):
      # calling the super fuction to access methods from parent class
        super(EmotionRecognitionModel, self).__init__()
      #using a pretrained vgg16 model
        self.vgg = models.vgg16(pretrained=True)
        # freezing the backbone
        for param in self.vgg.parameters():
            param.requires_grad = False
        #replacing the final layer with a custom classification layer
        self.vgg.classifier[6] = nn.Linear(self.vgg.classifier[6].in_features, num_classes)

    def forward(self, x):
        return self.vgg(x)

In [35]:
emotions=['anger', 'disgust', 'fear', 'happiness',
            'pleasant_surprise', 'sadness', 'neutral']
data_path="TESS Toronto emotional speech set data"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import zipfile as zf
with zf.ZipFile("/content/drive/MyDrive/TESS Toronto emotional speech set data-20250108T135115Z-001.zip","r") as zip_ref:
  zip_ref.extractall("/content")

In [ ]:
import os
import numpy as np
import torch
import librosa
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, random_split

In [ ]:
class EmotionDataset(Dataset):
  # defining a custom dataset class
    def __init__(self, data_path, emotions, transform=None):
        self.data_path = data_path #path of dataset
        self.emotions = emotions #list of emotions
        self.file_list = []
        self.labels = []
        self.transform = transform #object to perform transformations on data

        for idx, emotion in enumerate(emotions):
            emotion_folders = [f'YAF_{emotion}', f'OAF_{emotion}']
            for folder in emotion_folders:
                folder_path = os.path.join(data_path, folder)
                # joining data path with the name of the folder to get the complete folder path
                # e.g. "/content/TESS Toronto emotional speech set data/OAF_Fear/
                if os.path.exists(folder_path):
                    for file_name in os.listdir(folder_path):
                        file_path = os.path.join(folder_path, file_name)
                        # joining folder path with filenames to get the complete audio path
                        # "/content/TESS Toronto emotional speech set data/OAF_Fear/OAF_back_fear.wav"
                        self.file_list.append(file_path)
                        self.labels.append(idx)
                        # adding file paths and their respective labels to file_list and labels

    def __len__(self):
      #method to get the total number of audios
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        label = self.labels[idx]
        y, sr = librosa.load(file_path, sr=16000)
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
        max_length = 128
        pad_width = max_length - mel_spectrogram_db.shape[1]
        if pad_width > 0:
            mel_spectrogram_db = np.pad(mel_spectrogram_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mel_spectrogram_db = mel_spectrogram_db[:, :max_length]
        mel_spectrogram_3ch = np.repeat(mel_spectrogram_db[np.newaxis, :, :], 3, axis=0)
        return torch.tensor(mel_spectrogram_3ch, dtype=torch.float32), torch.tensor(label)

In [ ]:
class EmotionRecognitionModel(nn.Module):
  #defining the emotion recognition model
    def __init__(self, num_classes):
      # calling the super fuction to access methods from parent class
        super(EmotionRecognitionModel, self).__init__()
      #using a pretrained vgg16 model
        self.vgg = models.vgg16(pretrained=True)
        # freezing the backbone
        for param in self.vgg.parameters():
            param.requires_grad = False
        #replacing the final layer with a custom classification layer
        self.vgg.classifier[6] = nn.Linear(self.vgg.classifier[6].in_features, num_classes)

    def forward(self, x):
        return self.vgg(x)

In [ ]:
emotions=['anger', 'disgust', 'fear', 'happiness',
            'pleasant_surprise', 'sadness', 'neutral']
data_path="TESS Toronto emotional speech set data"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import zipfile as zf
with zf.ZipFile("/content/drive/MyDrive/TESS Toronto emotional speech set data-20250108T135115Z-001.zip","r") as zip_ref:
  zip_ref.extractall("/content")

In [ ]:
import os
import numpy as np
import torch
import librosa
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, random_split

In [ ]:
class EmotionDataset(Dataset):
  # defining a custom dataset class
    def __init__(self, data_path, emotions, transform=None):
        self.data_path = data_path #path of dataset
        self.emotions = emotions #list of emotions
        self.file_list = []
        self.labels = []
        self.transform = transform #object to perform transformations on data

        for idx, emotion in enumerate(emotions):
            emotion_folders = [f'YAF_{emotion}', f'OAF_{emotion}']
            for folder in emotion_folders:
                folder_path = os.path.join(data_path, folder)
                # joining data path with the name of the folder to get the complete folder path
                # e.g. "/content/TESS Toronto emotional speech set data/OAF_Fear/
                if os.path.exists(folder_path):
                    for file_name in os.listdir(folder_path):
                        file_path = os.path.join(folder_path, file_name)
                        # joining folder path with filenames to get the complete audio path
                        # "/content/TESS Toronto emotional speech set data/OAF_Fear/OAF_back_fear.wav"
                        self.file_list.append(file_path)
                        self.labels.append(idx)
                        # adding file paths and their respective labels to file_list and labels

    def __len__(self):
      #method to get the total number of audios
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        label = self.labels[idx]
        y, sr = librosa.load(file_path, sr=16000)
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
        max_length = 128
        pad_width = max_length - mel_spectrogram_db.shape[1]
        if pad_width > 0:
            mel_spectrogram_db = np.pad(mel_spectrogram_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mel_spectrogram_db = mel_spectrogram_db[:, :max_length]
        mel_spectrogram_3ch = np.repeat(mel_spectrogram_db[np.newaxis, :, :], 3, axis=0)
        return torch.tensor(mel_spectrogram_3ch, dtype=torch.float32), torch.tensor(label)

In [ ]:
class EmotionRecognitionModel(nn.Module):
  #defining the emotion recognition model
    def __init__(self, num_classes):
      # calling the super fuction to access methods from parent class
        super(EmotionRecognitionModel, self).__init__()
      #using a pretrained vgg16 model
        self.vgg = models.vgg16(pretrained=True)
        # freezing the backbone
        for param in self.vgg.parameters():
            param.requires_grad = False
        #replacing the final layer with a custom classification layer
        self.vgg.classifier[6] = nn.Linear(self.vgg.classifier[6].in_features, num_classes)

    def forward(self, x):
        return self.vgg(x)

In [ ]:
emotions=['anger', 'disgust', 'fear', 'happiness',
            'pleasant_surprise', 'sadness', 'neutral']
data_path="TESS Toronto emotional speech set data"

In [ ]:
dataset=EmotionDataset(data_path,emotions)
# initializing the dataset with data_path and list of emotions

In [ ]:
dataset.data_path

'TESS Toronto emotional speech set data'

In [ ]:
train_size=int(0.7 * len(dataset))
val_size=int(0.15 * len(dataset))
test_size=len(dataset) - train_size - val_size

In [ ]:
train_dataset, val_dataset, test_dataset = random_split(dataset,
                                                        [train_size, val_size, test_size])

In [ ]:
#creating dataloaders for batch processig
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=32)
test_loader=DataLoader(test_dataset, batch_size=32)

In [ ]:
model = EmotionRecognitionModel(num_classes=len(emotions))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 70.1MB/s]


In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    total_train_correct = 0
    total_train_samples = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        total_train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_train_samples += labels.size(0)

    train_accuracy = total_train_correct / total_train_samples
    print(f"Epoch [{epoch+1}/{num_epochs}] \n Train Loss: {train_loss/len(train_loader):.4f} \nTrain Accuracy: {train_accuracy:.4f}")

In [ ]:
dataset=EmotionDataset(data_path,emotions)
# initializing the dataset with data_path and list of emotions

In [ ]:
dataset.data_path

'TESS Toronto emotional speech set data'

In [ ]:
train_size=int(0.7 * len(dataset))
val_size=int(0.15 * len(dataset))
test_size=len(dataset) - train_size - val_size

In [ ]:
train_dataset, val_dataset, test_dataset = random_split(dataset,
                                                        [train_size, val_size, test_size])

In [ ]:
#creating dataloaders for batch processig
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=32)
test_loader=DataLoader(test_dataset, batch_size=32)

In [ ]:
model = EmotionRecognitionModel(num_classes=len(emotions))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 70.1MB/s]


In [ ]:
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    total_train_correct = 0
    total_train_samples = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        total_train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_train_samples += labels.size(0)

    train_accuracy = total_train_correct / total_train_samples
    print(f"Epoch [{epoch+1}/{num_epochs}] \n Train Loss: {train_loss/len(train_loader):.4f} \nTrain Accuracy: {train_accuracy:.4f}")

In [36]:
dataset=EmotionDataset(data_path,emotions)
# initializing the dataset with data_path and list of emotions

In [37]:
dataset.data_path

'TESS Toronto emotional speech set data'

In [38]:
train_size=int(0.7 * len(dataset))
val_size=int(0.15 * len(dataset))
test_size=len(dataset) - train_size - val_size

In [39]:
train_dataset, val_dataset, test_dataset = random_split(dataset,
                                                        [train_size, val_size, test_size])

In [40]:
#creating dataloaders for batch processig
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=32)
test_loader=DataLoader(test_dataset, batch_size=32)

In [41]:
model = EmotionRecognitionModel(num_classes=len(emotions))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 70.1MB/s]


In [42]:
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    total_train_correct = 0
    total_train_samples = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        total_train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_train_samples += labels.size(0)

    train_accuracy = total_train_correct / total_train_samples
    print(f"Epoch [{epoch+1}/{num_epochs}] \n Train Loss: {train_loss/len(train_loader):.4f} \nTrain Accuracy: {train_accuracy:.4f}")

Epoch [1/10] 
 Train Loss: 3.2177 
Train Accuracy: 0.4086
Epoch [2/10] 
 Train Loss: 1.6588 
Train Accuracy: 0.5857
Epoch [3/10] 
 Train Loss: 1.1338 
Train Accuracy: 0.6886
Epoch [4/10] 
 Train Loss: 0.8808 
Train Accuracy: 0.7271
Epoch [5/10] 
 Train Loss: 0.6812 
Train Accuracy: 0.8129
Epoch [6/10] 
 Train Loss: 0.5852 
Train Accuracy: 0.8257
Epoch [7/10] 
 Train Loss: 0.5501 
Train Accuracy: 0.8457
Epoch [8/10] 
 Train Loss: 0.5228 
Train Accuracy: 0.8386
Epoch [9/10] 
 Train Loss: 0.4026 
Train Accuracy: 0.8643
Epoch [10/10] 
 Train Loss: 0.4496 
Train Accuracy: 0.8686


In [44]:
torch.save({
        "epoch": epoch + 9,
        "model_state_dict": model.state_dict(),
    }, "/content/drive/MyDrive/SER_model.pth")